# HuecoEnv: GRPO Training Loop

This notebook contains the complete TRL (Transformer Reinforcement Learning) training loop for the HuecoEnv multi-agent simulation using GRPO. It adapts the Meta OpenEnv standard architecture to train an LLM to navigate the Scarcity Droughts.

In [ ]:
!pip install -Uq git+https://github.com/huggingface/trl.git git+https://github.com/meta-pytorch/OpenEnv.git trackio vllm==0.10.2 bitsandbytes

In [ ]:
import os
from huggingface_hub import notebook_login

notebook_login()

## Initialize the Environment

We initialize the `HuecoGymWrapper` which provides the standard OpenAI Gym API for our custom multi-agent logic.

In [ ]:
from train import HuecoGymWrapper

# We train specifically on the hard task so the agent experiences the Injector
task_name = "adaptive_survival"
wrapper = HuecoGymWrapper(task_name=task_name)

## Initialize Model and Tokenizer

We use a lightweight model for the hackathon baseline. You can scale this up to Llama-3.1-8B-Instruct if you have sufficient GPU memory.

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## GRPO Configuration

We configure the Group Relative Policy Optimization algorithm. We use vLLM in colocate mode to maximize generation efficiency during the rollout phase.

In [ ]:
from trl import GRPOConfig
from datasets import Dataset

# Create a dummy dataset (GRPO requires prompts to start the generation)
dataset = Dataset.from_dict({"prompt": ["Solve the environment."] * 500})

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    warmup_steps=20,
    num_generations=2,
    max_completion_length=128,
    max_prompt_length=1400,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.3,
    output_dir="huecoenv-grpo",
    logging_steps=1,
    save_steps=10,
)

## Training Loop & Rollout Function

The rollout function defines how the agent interacts with the environment during GRPO training.

In [ ]:
from trl import GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions
import json
import csv

# To track our survival rate over time
training_log = []

def rollout_func(prompts, trainer=None):
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    correctness_rewards = []

    for prompt_text in prompts:
        obs = wrapper.reset()
        done = False
        total_reward = 0.0
        
        # Generate actions using the model
        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        
        # In a real run, this parses the JSON action from the LLM and steps the environment
        try:
            action_text = rollout_outputs.get("text")
            obs, reward, done, info = wrapper.step()
            total_reward += reward
        except Exception:
            total_reward -= 1.0 # Penalty for bad formatting
            
        # Record the survival rate from the Environment Brain
        brain_info = wrapper.end_episode()
        survival_rate = brain_info.get("survival_rate", 0.0)
        training_log.append({"survival_rate": survival_rate})
        
        episode_prompt_ids.append(rollout_outputs["prompt_ids"])
        episode_completion_ids.append(rollout_outputs["completion_ids"])
        episode_logprobs.append(rollout_outputs["logprobs"])
        correctness_rewards.append(total_reward)

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "correct_reward": correctness_rewards,
    }

def reward_correct(completions, **kwargs):
    rewards = kwargs.get("correct_reward")
    return [float(r) for r in rewards] if rewards else [0.0] * len(completions)

trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    reward_funcs=[reward_correct],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

print("Starting TRL GRPO Training...")
trainer.train()

# Save data for the final graph
os.makedirs("data", exist_ok=True)
with open("data/training_adaptive_survival.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["episode", "survival_rate"])
    for idx, row in enumerate(training_log):
        writer.writerow([idx + 1, row["survival_rate"]])
        
print("Training Complete! Graph data saved.")